In [1]:
%pwd

'd:\\Tipto\\OmniChef-Nexus\\notebooks'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'd:\\Tipto\\OmniChef-Nexus'

In [4]:
import warnings
warnings.filterwarnings('ignore' , category = FutureWarning)

In [5]:
import pandas as pd 
df = pd.read_csv("data/all csv files/recipes_15k_samples.csv")

In [6]:
df.shape

(15698, 15)

In [7]:
import torch
from transformers import AutoModel , AutoProcessor
from transformers.image_utils import load_image
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [8]:
# load the embedding model with a specific revision version
model_name = "nvidia/llama-nemotron-embed-vl-1b-v2"
revision = "062ffaa1e3d24a8a50bd6a7ac7b8e54103e1f01d"

model = AutoModel.from_pretrained(
    model_name,
    revision = revision,
    dtype = torch.float16,
    trust_remote_code = True,
    attn_implementation = "flash_attention_2",
    device_map = "auto"
).eval()

Loading weights:   0%|          | 0/600 [00:00<?, ?it/s]

In [9]:
# make a helper function to change the model's modality
def prepare_processor(modality , embedding_model):
    """Prepare the model for different modality.

    Args:
        modality (str): can be 'text' , 'image' or 'image_text'
        embedding_model (_type_): embedding model

    Returns:
        _type_: (modality , embedding_model)
    """
    # Set max number of tokens (p_max_length) based on modality
    if modality == "image":
        p_max_length = 2048
    elif modality == "image_text":
        p_max_length = 10240
    elif modality == "text":
        p_max_length = 8192
    embedding_model.processor.p_max_length = p_max_length
    # Image specific settings(only matter if image is present)
    # Sets max number of tiles an image can be split. Each tile consumes 256 tokens.
    embedding_model.processor.max_input_tiles = 6
    # Enables an extra tile with the full image at lower resolution
    embedding_model.processor.use_thumbnail = True
    return modality , embedding_model

## Do embeddings for all images

In [10]:
# get all the image paths
all_image_paths = list(df['image_path'])
total_samples = len(all_image_paths)

print(f"[INFO]: total images: {total_samples}")

[INFO]: total images: 15698


In [11]:
# Pre-allocate the memory for all 15k samples
N = total_samples
D = 2048 # our embedding model produces 2048 dim vectors for each sample

# we are allowcating memory on cpu, cause our gpu can't hold all of them at once while calculating
embedding_image_all = torch.empty(size = (N , D) , dtype = torch.float16)
embedding_text_all = torch.empty(size = (N , D) , dtype = torch.float16)
embedding_image_text_all = torch.empty(size = (N , D) , dtype = torch.float16)

In [12]:
%%time 

from tqdm.auto import  tqdm
batch_size = 16


# start doing image only embeddings
for start_index in tqdm(range(0, total_samples, batch_size), desc = "Embedding Images"):
    end_index = min(start_index + batch_size, total_samples)
    
    # Load current batch
    batch_paths = all_image_paths[start_index : end_index]
    batch_images = [load_image(p) for p in batch_paths]
    
    with torch.inference_mode():
        # Generate embeddings
        batch_emb = model.encode_documents(images=batch_images)
        
        # Move to CPU and downcast
        batch_emb = batch_emb.to(device="cpu", dtype=torch.float16)
    
    # Store in the pre-allocated tensor
    embedding_image_all[start_index : end_index] = batch_emb
    
    # memory cleanup
    del batch_images
    del batch_emb
    torch.cuda.empty_cache()

print("Finished generating all embeddings!")

Embedding Images:   0%|          | 0/982 [00:00<?, ?it/s]

Finished generating all embeddings!
CPU times: total: 6h 4min 58s
Wall time: 2h 2min 47s


In [13]:
from safetensors.torch import save_file
tensor_dict = {
    "image_embeddings": embedding_image_all
}
# create the folder to store the embedding files
save_dir = "data/embedding_tensors"
os.makedirs(save_dir, exist_ok = True)

# Save to a single .safetensors file
file_path = os.path.join(save_dir , "all_recipes_image_embeddings.safetensors")
print(file_path)
save_file(tensor_dict, file_path)
print(f"[INFO]: Saved embeddings to {file_path}")

data/embedding_tensors\all_recipes_image_embeddings.safetensors
[INFO]: Saved embeddings to data/embedding_tensors\all_recipes_image_embeddings.safetensors


In [14]:
from safetensors.torch import load_file
image_embeddings = load_file("data/embedding_tensors/all_recipes_image_embeddings.safetensors")
image_embeddings

{'image_embeddings': tensor([[-1.2842,  1.0195,  2.2754,  ..., -2.4297, -0.4502,  0.2676],
         [-1.3857,  0.2522,  1.8154,  ...,  0.2947, -0.4600,  0.7261],
         [-0.1323, -0.9907,  1.1475,  ..., -1.5127, -1.7656, -0.5352],
         ...,
         [-1.0264,  1.6582,  1.0078,  ..., -0.0123, -1.6221,  0.1748],
         [-0.8638, -0.4421, -0.5532,  ..., -0.4114, -1.5869,  0.9663],
         [ 0.3906, -0.1992,  1.4854,  ..., -0.7246, -1.2148, -0.3403]],
        dtype=torch.float16)}

### Creating All the embeddings for texts

In [13]:
# set the modality to text
modality , model = prepare_processor(
    modality = 'text' , embedding_model = model
)

In [14]:
model.processor.p_max_length

8192

In [15]:
%%time 
from tqdm.auto import tqdm

batch_size = 32 

for start_index in tqdm(range(0, total_samples, batch_size), desc="Embedding Recipes Texts"):
    end_index = min(start_index + batch_size, total_samples)
    
    # Get all markdown text for current batch
    batch_texts = df['markdown_recipe'].iloc[start_index:end_index].tolist()
    
    with torch.inference_mode():
        # Generate embeddings using the text modality
        batch_emb = model.encode_documents(texts=batch_texts)
        
        # Move to CPU and downcast
        batch_emb = batch_emb.to(device="cpu", dtype=torch.float16)
    
    # Store in the TEXT pre-allocated tensor
    embedding_text_all[start_index : end_index] = batch_emb
    
    # Standard cleanup
    del batch_texts
    del batch_emb
    torch.cuda.empty_cache()

print("Finished generating all text embeddings!")

Embedding Recipes Texts:   0%|          | 0/491 [00:00<?, ?it/s]

Finished generating all text embeddings!
CPU times: total: 26min 37s
Wall time: 15min 15s


In [17]:
embedding_text_all.shape

torch.Size([15698, 2048])

In [18]:
embedding_text_all

tensor([[-0.9663,  0.4087,  2.3027,  ..., -2.0684, -0.4915, -1.5459],
        [-0.7783, -0.1185,  1.0674,  ...,  1.0156, -3.2324, -0.8735],
        [ 0.0726, -2.5488,  0.9277,  ..., -1.5098, -3.2383, -2.5820],
        ...,
        [-0.3103, -1.0654,  1.9043,  ...,  3.3379, -2.3750, -2.9453],
        [-0.6436, -2.5215, -0.5459,  ..., -0.4663, -1.9639, -2.1113],
        [-0.6685, -2.2266,  0.0639,  ...,  1.7480, -1.3525, -3.6328]],
       dtype=torch.float16)

In [19]:
from safetensors.torch import save_file
tensor_dict = {
    "text_embeddings": embedding_text_all
}
# create the folder to store the embedding files
save_dir = "data/embedding_tensors"
os.makedirs(save_dir, exist_ok = True)

# Save to a single .safetensors file
file_path = os.path.join(save_dir , "all_recipes_text_embeddings.safetensors")

save_file(tensor_dict, file_path)
print(f"[INFO]: Saved embeddings to {file_path}")

[INFO]: Saved embeddings to data/embedding_tensors\all_recipes_text_embeddings.safetensors


### Create embeddings for image and text combined

In [20]:
# set the modality
modality , model = prepare_processor(
    modality = 'image_text',
    embedding_model = model
)

In [21]:
model.processor.p_max_length

10240

In [27]:
from tqdm.auto import tqdm
batch_size = 16

for start_index in tqdm(range(0, total_samples , batch_size), desc=f"Embedding Modality: {modality}"):
    end_index = min(start_index + batch_size, total_samples)
    
    # Load both image and text for current batch
    batch_paths = all_image_paths[start_index : end_index]
    batch_images = [load_image(p) for p in batch_paths]
    batch_texts = df['markdown_recipe'].iloc[start_index:end_index].tolist()
    
    with torch.inference_mode():
        # creating the embeddings
        image_text_emb = model.encode_documents(images = batch_images, texts = batch_texts)
        # move the current from gpu to cpu
        image_text_emb = image_text_emb.to(device="cpu", dtype=torch.float16)
    
    # store the current batch
    embedding_image_text_all[start_index : end_index] = image_text_emb
    
    # remove current batch from gpu
    del image_text_emb , batch_images , batch_texts
    torch.cuda.empty_cache()

print("All embeddings generated and stored on CPU!")

Embedding Modality: image_text:   0%|          | 0/982 [00:00<?, ?it/s]

All embeddings generated and stored on CPU!


In [28]:
tensor_dict_image_text = {
    "image_text_embeddings": embedding_image_text_all
}
# create the folder to store the embedding files
save_dir = "data/embedding_tensors"
os.makedirs(save_dir, exist_ok = True)

# Save to a single .safetensors file
file_path = os.path.join(save_dir , "all_recipes_image_text_embeddings.safetensors")

save_file(tensor_dict_image_text, file_path)
print(f"[INFO]: Saved embeddings to {file_path}")

[INFO]: Saved embeddings to data/embedding_tensors\all_recipes_image_text_embeddings.safetensors


In [29]:
# save all image only , text only and image_text embedding in a single file

from safetensors.torch import load_file
image_embeddings = load_file("data/embedding_tensors/all_recipes_image_embeddings.safetensors")
text_embeddings = load_file("data/embedding_tensors/all_recipes_text_embeddings.safetensors")
image_text_embeddings = load_file("data/embedding_tensors/all_recipes_image_text_embeddings.safetensors")

In [30]:
image_embeddings

{'image_embeddings': tensor([[-1.2842,  1.0195,  2.2754,  ..., -2.4297, -0.4502,  0.2676],
         [-1.3857,  0.2522,  1.8154,  ...,  0.2947, -0.4600,  0.7261],
         [-0.1323, -0.9907,  1.1475,  ..., -1.5127, -1.7656, -0.5352],
         ...,
         [-1.0264,  1.6582,  1.0078,  ..., -0.0123, -1.6221,  0.1748],
         [-0.8638, -0.4421, -0.5532,  ..., -0.4114, -1.5869,  0.9663],
         [ 0.3906, -0.1992,  1.4854,  ..., -0.7246, -1.2148, -0.3403]],
        dtype=torch.float16)}

In [31]:
text_embeddings

{'text_embeddings': tensor([[-0.9663,  0.4087,  2.3027,  ..., -2.0684, -0.4915, -1.5459],
         [-0.7783, -0.1185,  1.0674,  ...,  1.0156, -3.2324, -0.8735],
         [ 0.0726, -2.5488,  0.9277,  ..., -1.5098, -3.2383, -2.5820],
         ...,
         [-0.3103, -1.0654,  1.9043,  ...,  3.3379, -2.3750, -2.9453],
         [-0.6436, -2.5215, -0.5459,  ..., -0.4663, -1.9639, -2.1113],
         [-0.6685, -2.2266,  0.0639,  ...,  1.7480, -1.3525, -3.6328]],
        dtype=torch.float16)}

In [32]:
image_text_embeddings

{'image_text_embeddings': tensor([[-0.9507,  1.1211,  2.6562,  ..., -2.3281, -0.3948,  0.0395],
         [-1.1504,  0.4902,  2.0195,  ...,  0.7817, -1.3008,  0.6104],
         [-0.1017, -1.1660,  0.9277,  ..., -1.5869, -2.0918, -0.8267],
         ...,
         [-0.7588,  1.2119,  1.5850,  ...,  0.6348, -1.9834, -0.1635],
         [-0.5239, -0.4646, -0.3879,  ..., -0.8237, -1.7998,  0.6162],
         [ 0.2712, -0.6646,  1.5234,  ..., -0.1637, -1.3701, -1.0488]],
        dtype=torch.float16)}

In [33]:
# create a dict
combined_data = {
    "image_embeddings": image_embeddings['image_embeddings'],
    "text_embeddings": text_embeddings['text_embeddings'],
    "image_text_embeddings": image_text_embeddings['image_text_embeddings']
}
combined_data

{'image_embeddings': tensor([[-1.2842,  1.0195,  2.2754,  ..., -2.4297, -0.4502,  0.2676],
         [-1.3857,  0.2522,  1.8154,  ...,  0.2947, -0.4600,  0.7261],
         [-0.1323, -0.9907,  1.1475,  ..., -1.5127, -1.7656, -0.5352],
         ...,
         [-1.0264,  1.6582,  1.0078,  ..., -0.0123, -1.6221,  0.1748],
         [-0.8638, -0.4421, -0.5532,  ..., -0.4114, -1.5869,  0.9663],
         [ 0.3906, -0.1992,  1.4854,  ..., -0.7246, -1.2148, -0.3403]],
        dtype=torch.float16),
 'text_embeddings': tensor([[-0.9663,  0.4087,  2.3027,  ..., -2.0684, -0.4915, -1.5459],
         [-0.7783, -0.1185,  1.0674,  ...,  1.0156, -3.2324, -0.8735],
         [ 0.0726, -2.5488,  0.9277,  ..., -1.5098, -3.2383, -2.5820],
         ...,
         [-0.3103, -1.0654,  1.9043,  ...,  3.3379, -2.3750, -2.9453],
         [-0.6436, -2.5215, -0.5459,  ..., -0.4663, -1.9639, -2.1113],
         [-0.6685, -2.2266,  0.0639,  ...,  1.7480, -1.3525, -3.6328]],
        dtype=torch.float16),
 'image_text_embedd

In [34]:
output_path = "data/embedding_tensors/all_recipes_embeddings.safetensors"
save_file(combined_data , output_path)
print(f"Successfully saved combined embeddings to {output_path}")

Successfully saved combined embeddings to data/embedding_tensors/all_recipes_embeddings.safetensors


In [37]:
# load the combined file
embeddings = load_file("data/embedding_tensors/all_recipes_embeddings.safetensors")
embeddings

{'image_embeddings': tensor([[-1.2842,  1.0195,  2.2754,  ..., -2.4297, -0.4502,  0.2676],
         [-1.3857,  0.2522,  1.8154,  ...,  0.2947, -0.4600,  0.7261],
         [-0.1323, -0.9907,  1.1475,  ..., -1.5127, -1.7656, -0.5352],
         ...,
         [-1.0264,  1.6582,  1.0078,  ..., -0.0123, -1.6221,  0.1748],
         [-0.8638, -0.4421, -0.5532,  ..., -0.4114, -1.5869,  0.9663],
         [ 0.3906, -0.1992,  1.4854,  ..., -0.7246, -1.2148, -0.3403]],
        dtype=torch.float16),
 'image_text_embeddings': tensor([[-0.9507,  1.1211,  2.6562,  ..., -2.3281, -0.3948,  0.0395],
         [-1.1504,  0.4902,  2.0195,  ...,  0.7817, -1.3008,  0.6104],
         [-0.1017, -1.1660,  0.9277,  ..., -1.5869, -2.0918, -0.8267],
         ...,
         [-0.7588,  1.2119,  1.5850,  ...,  0.6348, -1.9834, -0.1635],
         [-0.5239, -0.4646, -0.3879,  ..., -0.8237, -1.7998,  0.6162],
         [ 0.2712, -0.6646,  1.5234,  ..., -0.1637, -1.3701, -1.0488]],
        dtype=torch.float16),
 'text_embedd